# Testing MCP Client with Typescript MCP Server on Amazon Bedrock AgentCore Runtime

# 在 Amazon Bedrock AgentCore Runtime 上使用 TypeScript MCP 服务器测试 MCP 客户端

## Overview

## 概述

In this tutorial we will learn how to host a TypeScript-based MCP (Model Context Protocol) server using the Amazon Bedrock AgentCore runtime environment.

在本教程中，我们将学习如何使用 Amazon Bedrock AgentCore 运行时环境托管基于 TypeScript 的 MCP（模型上下文协议）服务器。

### Tutorial Details

### 教程详情

| Information         | Details                                                   |
|:--------------------|:----------------------------------------------------------|
| Tutorial type       | Hosting typescript MCP server                             |
| Tool type           | MCP server                                                |
| Tutorial components | Hosting typescript MCP server on AgentCore Runtime        |
| Tutorial vertical   | Cross-vertical                                            |
| Example complexity  | Easy                                                      |
| SDK used            | Anthropic's typescript SDK for MCP                        |

| 信息                 | 详情                                                       |
|:--------------------|:----------------------------------------------------------|
| 教程类型             | 托管 TypeScript MCP 服务器                                 |
| 工具类型             | MCP 服务器                                                 |
| 教程组件             | 在 AgentCore Runtime 上托管 TypeScript MCP 服务器          |
| 教程行业             | 跨行业                                                     |
| 示例复杂度           | 简单                                                       |
| 使用的 SDK          | Anthropic 的 TypeScript MCP SDK                           |

### Tutorial Overview

### 教程概览

1. The AgentCore Runtime authentication will use Amazon Cognito to provide JWT tokens for accessing our deployed MCP server.

1. AgentCore Runtime 身份验证将使用 Amazon Cognito 提供 JWT 令牌来访问已部署的 MCP 服务器。

2. The mcp server is written in typescript and will be [deployed using custom flow](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html)

2. MCP 服务器使用 TypeScript 编写，将[使用自定义流程部署](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html)

3. The mcp client is written in python.
   _Note the mcp client can be written in any language._

3. MCP 客户端使用 Python 编写。
   _注意：MCP 客户端可以使用任何语言编写。_

## Prerequisites

## 前提条件

To execute this tutorial you will need:
- Node.js v22 or later  (MCP server)
- Python 3.10+ (MCP client)
- Docker (for containerization)
- Amazon ECR (Elastic Container Registry) for storing Docker images
- AWS account with access to Bedrock AgentCore
- MCP (Model Context Protocol) library
- Docker running

要执行本教程，您需要：
- Node.js v22 或更高版本（MCP 服务器）
- Python 3.10+（MCP 客户端）
- Docker（用于容器化）
- Amazon ECR（弹性容器注册表）用于存储 Docker 镜像
- 具有 Bedrock AgentCore 访问权限的 AWS 账户
- MCP（模型上下文协议）库
- 运行中的 Docker

In [ ]:
#!uv add -r requirements.txt --active

## Understanding MCP (Model Context Protocol)

## 理解 MCP（模型上下文协议）

MCP is a protocol that allows AI models to securely access external data and tools. Key concepts:

MCP 是一种允许 AI 模型安全访问外部数据和工具的协议。关键概念：

* **Tools**: Functions that the AI can call to perform actions
* **Prompts**: Prompts allow servers to provide structured messages and instructions for interacting with LLM
* **Streamable HTTP**: Transport protocol used by AgentCore Runtime
* **Session Isolation**: Each client gets isolated sessions via `Mcp-Session-Id` header
* **Stateless Operation**: Servers must support stateless operation for scalability

* **工具（Tools）**：AI 可以调用的用于执行操作的函数
* **提示（Prompts）**：提示允许服务器提供结构化消息和指令以与 LLM 交互
* **可流式 HTTP（Streamable HTTP）**：AgentCore Runtime 使用的传输协议
* **会话隔离（Session Isolation）**：每个客户端通过 `Mcp-Session-Id` 头获得隔离的会话
* **无状态操作（Stateless Operation）**：服务器必须支持无状态操作以实现可扩展性

AgentCore Runtime expects MCP servers to be hosted on `0.0.0.0:8000/mcp` as the default path.

AgentCore Runtime 期望 MCP 服务器托管在 `0.0.0.0:8000/mcp` 作为默认路径。

## Step 1: Setting up Amazon Cognito for Authentication

## 步骤 1：设置 Amazon Cognito 进行身份验证

AgentCore Runtime requires authentication. We'll use Amazon Cognito to provide JWT tokens for accessing our deployed MCP server.

AgentCore Runtime 需要身份验证。我们将使用 Amazon Cognito 提供 JWT 令牌来访问已部署的 MCP 服务器。

In [ ]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import create_agentcore_role, setup_cognito_user_pool

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

## Step 2: Create IAM Execution Role

## 步骤 2：创建 IAM 执行角色

Before starting, let's create an IAM role for our AgentCore Runtime. This role provides the necessary permissions for the runtime to operate.

在开始之前，让我们为 AgentCore Runtime 创建一个 IAM 角色。此角色提供运行时操作所需的必要权限。

In [ ]:
tool_name = "mcp_server_ac"
print(f"Creating IAM role for {tool_name}...")
agentcore_iam_role = create_agentcore_role(agent_name=tool_name)
print(f"IAM role created ✓")
print(f"Role ARN: {agentcore_iam_role['Role']['Arn']}")

## Step 3: Creating MCP Server

## 步骤 3：创建 MCP 服务器

Now let's create our a typescript MCP server with two simple tools and one prompt. Navigate to the src folder under this tutorial.

现在让我们创建一个包含两个简单工具和一个提示的 TypeScript MCP 服务器。导航到本教程下的 src 文件夹。

1. Install dependencies

1. 安装依赖

```
npm install
```

2. Set up AWS credentials

2. 设置 AWS 凭证

```
aws configure
export AWS_ACCESS_KEY_ID=your_access_key
export AWS_SECRET_ACCESS_KEY=your_secret_key
export AWS_REGION=us-east-1
```

3. Start server (running locally)

3. 启动服务器（本地运行）

```
npm run start
```

## Step 4: MCP Server Deployment through Docker

## 步骤 4：通过 Docker 部署 MCP 服务器

Note: This are manual steps for deploying an agent or mcp server without the starter toolkit

注意：这些是不使用启动工具包部署代理或 MCP 服务器的手动步骤

https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

1. Create ECR Repository

1. 创建 ECR 仓库

```
aws ecr create-repository --repository-name mcp-server --region us-east-1
```

2. Build and Push Image to ECR

2. 构建并推送镜像到 ECR

```
# Get login token
# 获取登录令牌
aws ecr get-login-password --region us-east-1 | \
  docker login --username AWS --password-stdin [account-id].dkr.ecr.us-east-1.amazonaws.com

docker buildx --platform linux/arm64 \
  -t [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest --push .
```

3. Deploy to Bedrock AgentCore

3. 部署到 Bedrock AgentCore

    - Go to AWS Console → Bedrock → AgentCore → Create Agent
    - Choose MCP as the protocol
    - Configure Agent Runtime:
        - Image URI: [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest
        - Set IAM Permissions for Bedrock model access
        - Deploy and test in the Agent Sandbox
    - For Discovery url: Select the url from above, cognito_config['discovery_url']
    - For Client id: Select the client id from above, cognito_config['client_id']
    - For execution role: Select the arn from above, agentcore_iam_role['Role']['Arn']

    - 进入 AWS 控制台 → Bedrock → AgentCore → 创建代理
    - 选择 MCP 作为协议
    - 配置代理运行时：
        - 镜像 URI：[account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest
        - 设置 Bedrock 模型访问的 IAM 权限
        - 在代理沙箱中部署和测试
    - Discovery URL：选择上面的 URL，cognito_config['discovery_url']
    - Client ID：选择上面的客户端 ID，cognito_config['client_id']
    - 执行角色：选择上面的 ARN，agentcore_iam_role['Role']['Arn']

## Step 5: Storing Configuration for Remote Access

## 步骤 5：存储远程访问配置

Before we can invoke our deployed MCP server, let's store the Agent ARN (fetch the arn from Step 4) and Cognito configuration in AWS Systems Manager Parameter Store and AWS Secrets Manager for easy retrieval:

在我们能够调用已部署的 MCP 服务器之前，让我们将 Agent ARN（从步骤 4 获取）和 Cognito 配置存储在 AWS Systems Manager Parameter Store 和 AWS Secrets Manager 中，以便于检索：

In [ ]:
import boto3
import json

boto_session = Session()
region = boto_session.region_name

ssm_client = boto3.client('ssm', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name='mcp_server/cognito/credentials',
        Description='Cognito credentials for MCP server',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId='mcp_server/cognito/credentials',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials updated in Secrets Manager")

 # NOTE: Add your agent arn that you created in Step 4
agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime/agent_arn',
    Value="Add your agent arn that you created in step 4", 
    Type='String',
    Description='Agent ARN for MCP server',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

## Step 6: Creating Remote Testing Client

## 步骤 6：创建远程测试客户端

Now let's create a client to test our deployed MCP server. This client will retrieve the necessary credentials from AWS and connect to the deployed server:

现在让我们创建一个客户端来测试我们部署的 MCP 服务器。此客户端将从 AWS 检索必要的凭证并连接到已部署的服务器：

In [ ]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")
     
        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    

    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Step 7: Testing Your Deployed MCP Server

## 步骤 7：测试您部署的 MCP 服务器

Let's test our deployed MCP server using the remote client:

让我们使用远程客户端测试已部署的 MCP 服务器：

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python my_mcp_client_remote.py

## Step 8: Invoking MCP Tools Remotely

## 步骤 8：远程调用 MCP 工具

Now let's create an enhanced client that not only lists tools but also invokes them to demonstrate the full MCP functionality:

现在让我们创建一个增强版客户端，它不仅可以列出工具，还可以调用它们来演示完整的 MCP 功能：

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")
                
                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)
                
                try:
                    print("\n➕ Testing add(5, 3)...")
                    add_result = await session.call_tool(
                        name="add",
                        arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                try:
                    print("\n✖️  Testing subtract(10, 2)...")
                    substract_result = await session.call_tool(
                        name="subtract",
                        arguments={"a": 10, "b": 2}
                    )
                    print(f"   Result: {substract_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                print("\n✅ MCP tool testing completed!")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Test Tool Invocation

## 测试工具调用

Let's test our MCP tools by actually invoking them:

让我们通过实际调用来测试我们的 MCP 工具：

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

## Next Steps

## 后续步骤

Now that you have successfully deployed an MCP server to AgentCore Runtime, you can:

现在您已成功将 MCP 服务器部署到 AgentCore Runtime，您可以：

1. **Add More Tools**: Extend your MCP server with additional tools
2. **Custom Authentication**: Implement custom JWT authorizers
3. **Integration**: Integrate with other AgentCore services

1. **添加更多工具**：使用额外的工具扩展您的 MCP 服务器
2. **自定义身份验证**：实现自定义 JWT 授权器
3. **集成**：与其他 AgentCore 服务集成

# 🎉 Congratulations!

# 🎉 恭喜！

You have successfully:

您已成功完成：

✅ **Created a typescript MCP server** with custom tools
✅ **Set up authentication** with Amazon Cognito
✅ **Deployed to AWS** using AgentCore Runtime
✅ **Invoked remotely** with proper authentication
✅ **Learned MCP concepts** and best practices

✅ **创建了 TypeScript MCP 服务器**，包含自定义工具
✅ **设置了身份验证**，使用 Amazon Cognito
✅ **部署到 AWS**，使用 AgentCore Runtime
✅ **远程调用**，使用正确的身份验证
✅ **学习了 MCP 概念**和最佳实践

Your MCP server is now running on Amazon Bedrock AgentCore Runtime and ready for production use!

您的 MCP 服务器现在正在 Amazon Bedrock AgentCore Runtime 上运行，并已准备好用于生产环境！